# 02 - API Integration with Ollama (Python)

This lab notebook focuses on **robust programmatic integration** with the Ollama REST API.

Topics:
- Health checks
- Listing installed models
- `/api/generate` and `/api/chat`
- Streaming responses (JSONL)
- Error handling and timeouts


In [1]:
import os
import requests

# Resolve Ollama endpoint depending on environment
BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
print('Using Ollama at:', BASE_URL)

# Diagnostic check
try:
    r = requests.get(BASE_URL, timeout=3)
    print('Server reachable. Status:', r.status_code)
    try:
        tags = requests.get(f"{BASE_URL}/api/tags", timeout=3)
        print('Tags endpoint status:', tags.status_code)
    except Exception as e:
        print('Tags endpoint check failed:', e)
except Exception as e:
    print('Server not reachable:', e)


Using Ollama at: http://host.docker.internal:11434
Server reachable. Status: 200
Tags endpoint status: 200


In [2]:
import json
import requests
from typing import Iterator, Dict, Any, List


In [3]:
BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')

def health_check() -> bool:
    try:
        r = requests.get(BASE_URL, timeout=3)
        print('Status:', r.status_code)
        return r.ok
    except requests.RequestException as e:
        print('Health check failed:', e)
        return False

health_check()



Status: 200


True

In [4]:
def list_models() -> List[Dict[str, Any]]:
    r = requests.get(f'{BASE_URL}/api/tags', timeout=10)
    r.raise_for_status()
    return r.json().get('models', [])

models = list_models()
print('Installed models:', [m.get('name') for m in models])



Installed models: ['llama3.2:3b', 'gemma3:4b', 'gpt-oss:20b', 'gemma:2b', 'llama3.2:latest']


In [5]:
def generate(prompt: str, model: str = 'llama3.2:3b', temperature: float = 0.2) -> str:
    payload = {'model': model, 'prompt': prompt, 'temperature': temperature, 'stream': False}
    r = requests.post(f'{BASE_URL}/api/generate', json=payload, timeout=120)
    r.raise_for_status()
    return r.json().get('response', '')

print(generate('Explain overfitting in machine learning in 5 bullet points.'))



Here are five bullet points explaining overfitting in machine learning:

• **Definition**: Overfitting occurs when a model is too complex and perfectly fits the training data, but poorly generalizes to new, unseen data. This means the model is too specialized to the training data and fails to capture the underlying patterns and relationships.

• **Consequences**: Overfitting can lead to poor performance on test data, low accuracy, and high error rates. It can also lead to the model overestimating its ability to generalize to new data, which can result in poor decision-making and misclassification.

• **Causes**: Overfitting can occur when the model is too complex, the training data is too small, or the model is not regularized. Other causes include using too many features, not enough regularization, or using a model with too many layers.

• **Detection**: Overfitting can be detected by checking the model's performance on the training and test data. If the model performs well on the tra

In [6]:
def chat(messages: List[Dict[str, str]], model: str = 'llama3.2:3b', temperature: float = 0.2) -> str:
    payload = {'model': model, 'messages': messages, 'temperature': temperature, 'stream': False}
    r = requests.post(f'{BASE_URL}/api/chat', json=payload, timeout=120)
    r.raise_for_status()
    return r.json().get('message', {}).get('content', '')

msgs = [
    {'role': 'system', 'content': 'You are a concise teaching assistant.'},
    {'role': 'user', 'content': 'What is the bias-variance tradeoff?'}
]
print(chat(msgs))



The bias-variance tradeoff is a fundamental concept in machine learning and statistics.

**Bias-Variance Tradeoff:**

In the context of regression models, the bias-variance tradeoff refers to the tradeoff between two types of errors:

1. **Bias**: This occurs when a model is too simple and underfits the data, resulting in too much variance (error) in the predicted values. Biased models tend to have a consistent but inaccurate prediction.
2. **Variance**: This occurs when a model is too complex and overfits the data, resulting in too much variability (error) in the predicted values. Varied models tend to have a poor fit to the data.

**The Tradeoff:**

The ideal model should have a balance between bias and variance. When a model is too simple (biased), it may not capture the underlying patterns in the data, leading to poor predictions. On the other hand, when a model is too complex (varied), it may fit the noise in the data rather than the underlying patterns, leading to overfitting.

*

In [7]:
def generate_stream(prompt: str, model: str = 'llama3.2:3b') -> Iterator[str]:
    payload = {'model': model, 'prompt': prompt, 'stream': True}
    with requests.post(f'{BASE_URL}/api/generate', json=payload, stream=True, timeout=300) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if not line:
                continue
            obj = json.loads(line.decode('utf-8'))
            yield obj.get('response', '')

text = ''
for chunk in generate_stream('Write a short explanation of transformers for engineers.'):
    text += chunk
print(text)



**Transformers: A Fundamental Concept in Electrical Engineering**

Transformers are a crucial component in electrical power systems, used to step up or step down voltage levels in order to efficiently transmit power over long distances. The transformer is a type of electrical device that transfers electrical energy from one circuit to another through electromagnetic induction.

**Principle of Operation**

A transformer consists of two or more coils of wire, known as primary and secondary coils, which are wrapped around a common magnetic core. The primary coil is connected to the power source, while the secondary coil is connected to the load. When an alternating current (AC) flows through the primary coil, a magnetic field is generated, which induces a voltage in the secondary coil.

**Types of Transformers**

There are several types of transformers, including:

* **Step-up transformer**: Increases the voltage level to transmit power over long distances.
* **Step-down transformer**: De